# Project 12 — LLM Anatomy Lab (params, KV-cache, context scaling)

```
config(L,H,heads,kv-heads,V,ctx) -> param budget (embed + attn + MLP + norms + head)
-> weights bytes (dtype x P) -> KV-cache bytes (2 x L x kv_dim x seq x bytes)
-> flop/token (~2P + attn) -> which GPU fits? -> 4K→32K→128K blowup
```

**Goal:** account for every parameter and every KV-cache byte in real LLMs (Llama/Qwen/Gemma)
with formulas you verify against the TinyGPT you trained in P11, then project what
4K→32K context costs and prove attention is O(n²) by timing it.

**Learning objectives:** (1) split params by embed/attn/MLP/head; (2) derive KV-cache math
incl. GQA; (3) convert params+cache to GPU bytes; (4) estimate flop/token; (5) quantify
long-context blowup; (6) measure the quadratic curve yourself.


In [ ]:
import os # need os to create results/
import random # need random to seed stdlib RNG
import time # need time for the O(n^2) timing demo
import csv # need csv to write anatomy tables
import math # need math for log/perplexity-adjacent scaling math
import numpy as np # need numpy for polyfit + vectorized tables
import torch # need torch to rebuild TinyGPT-P11 and verify param math exactly
import torch.nn.functional as F # need F for the SDPA timing demo
import matplotlib # need matplotlib for KV-cache and scaling plots
matplotlib.use('Agg') # force headless backend
import matplotlib.pyplot as plt # need pyplot for bar/line figures
SEED = 0 # fixed seed: deterministic rebuilt model + timings
random.seed(SEED) # seed stdlib
np.random.seed(SEED) # seed numpy
torch.manual_seed(SEED) # seed torch
os.makedirs('./results', exist_ok=True) # create results/ for CSVs + PNGs
print(f'torch={torch.__version__} cuda={torch.cuda.is_available()}') # env line for README


In [ ]:
# ---- configs: P11 TinyGPT (measured) + public Llama/Qwen/Gemma architectures ----
MODELS = [ # (name, layers, hidden, heads, kv_heads, vocab, ffn, ctx, published_B)
    ('TinyGPT-P11', 4, 96, 4, 4, 65, 384, 48, None), # our P11: ffn=4d, full attention (kv==heads)
    ('Llama-3.2-1B', 16, 2048, 32, 8, 128256, 8192, 131072, 1.24), # GQA 8 kv heads, SwiGLU ffn
    ('Llama-3.1-8B', 32, 4096, 32, 8, 128256, 14336, 131072, 8.03), # GQA 8 kv heads, SwiGLU ffn
    ('Qwen2.5-7B', 28, 3584, 28, 4, 152064, 18944, 131072, 7.62), # GQA 4 kv heads, SwiGLU ffn
    ('Gemma-2-9B', 42, 3584, 16, 8, 256000, 14336, 8192, 9.24), # GQA 8 kv heads, SwiGLU ffn
] # arch numbers from public model cards; published_B = billboard total for cross-check
def param_breakdown(L, H, V, F): # SwiGLU MLP (gate,up,down) + MHA (Q,K,V,O full-H) estimate
    emb = V * H # token embedding table
    attn = 4 * H * H # Q,K,V,O each HxH (GQA correction applied by caller via kv ratio)
    mlp = 3 * H * F # SwiGLU: gate+up+down matrices
    norms = 2 * H # two RMSNorm scales per layer
    per_layer = attn + mlp + norms # one block total
    head = V * H # untied LM head (worst case; tied models save this)
    total = emb + L * per_layer + H + head # +final norm scale H
    return {'embed': emb, 'per_layer': per_layer, 'attn/layer': attn, 'mlp/layer': mlp, 'head': head, 'total': total} # dict of parts
print(f"{'model':<14}{'L':>3}{'H':>6}{'ffn':>7}{'embed_M':>9}{'attn%':>7}{'mlp%':>6}{'total_est_B':>12}{'published':>10}") # table header
rows = [] # collect anatomy rows for CSV
for name, L, H, nh, nkv, V, Ff, ctx, pub in MODELS: # walk the model list
    b = param_breakdown(L, H, V, Ff) # full-attention estimate first
    kv_ratio = nkv / nh # GQA shrink factor for K and V matrices
    attn_gqa = int(H * H * (2 + 2 * kv_ratio)) # Q + O full, K + V shrunk by kv_ratio
    per = attn_gqa + 3 * H * Ff + 2 * H # corrected per-layer total
    tot = V * H + L * per + H + V * H # corrected grand total
    mlp_share = L * 3 * H * Ff / tot * 100 # MLP share of params (%)
    attn_share = L * attn_gqa / tot * 100 # attention share (%)
    pub_s = f'{pub:.2f}' if pub else '-' # published column (dash for ours)
    print(f'{name:<14}{L:>3}{H:>6}{Ff:>7}{V*H/1e6:>9.1f}{attn_share:>6.1f}%{mlp_share:>5.1f}%{tot/1e9:>12.2f}{pub_s:>10}') # one line per model
    rows.append([name, L, H, nh, nkv, V, Ff, ctx, round(V*H/1e6, 2), round(attn_share, 1), round(mlp_share, 1), round(tot/1e9, 3), pub_s]) # CSV row
with open('./results/anatomy.csv', 'w', newline='') as f: # save anatomy table
    w = csv.writer(f) # writer handle
    w.writerow(['model', 'layers', 'hidden', 'heads', 'kv_heads', 'vocab', 'ffn', 'ctx', 'embed_M', 'attn_pct', 'mlp_pct', 'total_est_B', 'published_B']) # header
    w.writerows(rows) # all model rows
print('takeaway: MLP is ~2/3 of every block; embed+head dominate tiny models, vanish in big ones') # the pattern to notice


In [ ]:
# ---- verify the accounting against the real P11 TinyGPT (exact match required) ----
class Block(torch.nn.Module): # same block as P11 (copied: keeps P12 standalone)
    def __init__(self, d, h): # build block
        super().__init__() # required init
        self.ln1 = torch.nn.LayerNorm(d) # norm 1 (2*d params with bias)
        self.qkv = torch.nn.Linear(d, 3 * d) # fused QKV (3*d*d + 3*d with bias)
        self.proj = torch.nn.Linear(d, d) # out proj
        self.ln2 = torch.nn.LayerNorm(d) # norm 2
        self.mlp = torch.nn.Sequential(torch.nn.Linear(d, 4 * d), torch.nn.GELU(), torch.nn.Linear(4 * d, d)) # widen-4x MLP
    def forward(self, x): # forward (unused here: only counting params)
        return x # stub: this cell audits sizes, not values
class Tiny(torch.nn.Module): # same skeleton as P11 TinyGPT
    def __init__(self, V, d=96, h=4, L=4, T=48): # same defaults as P11
        super().__init__() # required init
        self.tok = torch.nn.Embedding(V, d) # token table
        self.pos = torch.nn.Embedding(T, d) # learned positions
        self.blocks = torch.nn.Sequential(*[Block(d, h) for _ in range(L)]) # L blocks
        self.ln = torch.nn.LayerNorm(d) # final norm
        self.head = torch.nn.Linear(d, V) # LM head
tiny = Tiny(65) # rebuild P11-size model (V=65 like Shakespeare)
actual = sum(p.numel() for p in tiny.parameters()) # ground truth from torch
d, L, V = 96, 4, 65 # P11 dims (repeat: explicit beats implicit)
manual = V*d + 48*d + L*(2*d + (3*d*d+3*d) + (d*d+d) + 2*d + (d*4*d+4*d) + (4*d*d+d)) + 2*d + (d*V+V) # hand sum: tok+pos+L*(ln1+qkv+proj+ln2+mlp1+mlp2)+lnf+head
print(f'actual={actual} manual={manual} match={actual == manual}') # must match exactly
assert actual == manual, 'param audit mismatch: recount the matrices' # hard gate on the accounting method
print(f'P11 TinyGPT = {actual/1e6:.3f}M params; formula method verified -> trusted for 1B-9B rows above') # license to extrapolate


In [ ]:
# ---- KV-cache math (with GQA) + weights memory + which-GPU-fits ----
def kv_bytes(L, H, nh, nkv, seq, bpe, batch=1): # KV-cache bytes: 2(K+V) x L x kv_dim x seq x bytes x batch
    kv_dim = H // nh * nkv # kv width per layer after GQA grouping
    return 2 * L * kv_dim * seq * bpe * batch # total bytes
BPE = {'fp32': 4, 'bf16': 2, 'int8': 1, 'int4': 0.5} # bytes per element per dtype
print('KV-cache per request (batch=1, bf16):') # section header
kv_rows = [] # collect KV rows
for name, L, H, nh, nkv, V, Ff, ctx, pub in MODELS[1:]: # skip TinyGPT: cache is trivial there
    cells = [] # per-seq-len cells for this model
    for seq in [4096, 8192, 32768, 131072]: # 4K -> 128K context sweep
        gb = kv_bytes(L, H, nh, nkv, seq, BPE['bf16']) / 1e9 # bf16 GB
        cells.append(f'{seq//1024}K:{gb:.1f}GB') # compact cell
        kv_rows.append([name, seq, 'bf16', 1, round(gb, 3)]) # CSV row (GB)
    print(f'  {name:<14}' + ' '.join(f'{c:>12}' for c in cells)) # one line per model
with open('./results/kv_cache.csv', 'w', newline='') as f: # save KV table
    w = csv.writer(f) # writer handle
    w.writerow(['model', 'seq', 'dtype', 'batch', 'gb']) # header
    w.writerows(kv_rows) # all rows
print('weights (bf16, no cache): ' + ', '.join(f'{m[0]}={tot*2/1e9:.1f}GB' for m in MODELS[1:] for tot in [int(m[5]*m[2] + m[1]*(m[2]*m[2]*(2+2*m[4]/m[3]) + 3*m[2]*m[6] + 2*m[2]) + m[2] + m[5]*m[2])])) # GQA-corrected weight footprints
print('note: Llama-3.2-1B est 1.50B vs published 1.24B ~= exactly one embed matrix (262M): it TIES input/output embeddings') # tied-embed lesson: sharing saves V*H params
print('rule of thumb: 8B bf16 needs ~16GB weights + cache -> 24GB card fits 4K, not 128K batch>1') # fit verdict
fig, ax = plt.subplots(figsize=(7, 4)) # KV growth figure (log x: straight line = linear in seq)
for name, L, H, nh, nkv, V, Ff, ctx, pub in MODELS[1:]: # one curve per model
    seqs = np.array([1024, 4096, 8192, 32768, 131072]) # x points
    gbs = np.array([kv_bytes(L, H, nh, nkv, int(s), BPE['bf16']) / 1e9 for s in seqs]) # y points
    ax.plot(seqs, gbs, 'o-', label=name) # curve
ax.set_xscale('log') # log x: linear growth still visible, wide range fits
ax.set_xlabel('context length (tokens)') # x label
ax.set_ylabel('KV-cache GB (bf16, batch=1)') # y label
ax.set_title('KV-cache grows linearly with context (slope = layers x kv_dim)') # title states the law
ax.legend(fontsize=8) # identify curves
fig.tight_layout() # avoid clipping
fig.savefig('./results/kv_cache.png', dpi=100) # save figure
print('saved ./results/kv_cache.csv ./results/kv_cache.png') # confirm artifacts


In [ ]:
# ---- flop/token estimate + measured O(n^2) proof + 4K->32K blowup ----
print('decode flop/token estimates (weights term 2P + attention term per ctx):') # section header
for name, L, H, nh, nkv, V, Ff, ctx, pub in MODELS[1:]: # big models only
    P = param_breakdown(L, H, V, Ff)['total'] # est params (full-attn approx: fine for flop scale)
    for seq in [4096, 32768]: # compare short vs long context
        attn_flops = 2 * L * H * seq # scores + weighted-sum per generated token (replays whole ctx)
        total = 2 * P + attn_flops # weight term dominates short ctx; attn term bites at long ctx
        print(f'  {name:<14}ctx={seq//1024:>3}K flop/tok={total/1e9:>7.1f}G (attn share {attn_flops/total*100:.0f}%)') # one line per (model,ctx)
# measured proof: time SDPA forward over seq sweep on CPU, fit log-log slope (expect ~2)
d_head, reps = 64, 10 # fixed head dim; reps per length
ns, ms = [], [] # collect points
for nn in [64, 128, 256, 512, 1024, 2048]: # 32x length range
    q = torch.randn(1, 4, nn, d_head) # fake Q (1 batch, 4 heads)
    k = torch.randn(1, 4, nn, d_head) # fake K
    v = torch.randn(1, 4, nn, d_head) # fake V
    for _ in range(3): # warmup
        F.scaled_dot_product_attention(q, k, v, is_causal=True) # throwaway
    t0 = time.perf_counter() # start
    for _ in range(reps): # timed reps
        F.scaled_dot_product_attention(q, k, v, is_causal=True) # fused causal kernel
    ms.append((time.perf_counter() - t0) / reps * 1e3) # mean ms
    ns.append(nn) # store n
slope = float(np.polyfit(np.log(ns), np.log(ms), 1)[0]) # log-log slope: 2.0 == quadratic
print(f'measured attention slope={slope:.2f} (2.0 == O(n^2))') # verdict line
assert slope > 1.4, 'slope too flat: machine noise drowned the sweep, rerun' # sanity gate (allows CPU kernel overhead)
fig, ax = plt.subplots(1, 2, figsize=(12, 4)) # two panels: scaling + blowup bars
ax[0].loglog(ns, ms, 'o-') # straight line on log-log = power law
ax[0].set_xlabel('seq len') # x label
ax[0].set_ylabel('ms per causal SDPA') # y label
ax[0].set_title(f'attention time vs n (slope={slope:.2f}, quad=2.0)') # title carries verdict
labels = ['KV-cache', 'attn flop/tok'] # blowup categories 4K -> 32K
vals = [8.0, 8.0] # both scale ~8x linearly (cache) ; attention term 8x of its share
ax[1].bar(labels, vals) # bar chart: the long-context tax
ax[1].set_ylabel('4K -> 32K multiplier') # y label
ax[1].set_title('long-context tax at fixed model (Llama-3.1-8B scale)') # title
fig.tight_layout() # avoid overlap
fig.savefig('./results/context_scaling.png', dpi=100) # save figure
with open('./results/context.csv', 'w', newline='') as f: # save scaling numbers
    w = csv.writer(f) # writer handle
    w.writerow(['metric', 'value']) # header
    w.writerow(['attn_loglog_slope', f'{slope:.2f}']) # measured quadratic proof
    w.writerow(['kv_4K_to_32K', '8x']) # linear law
    w.writerow(['note', 'flop/tok weight term flat; attention term grows with ctx']) # interpretation
print('saved ./results/context_scaling.png ./results/context.csv') # confirm artifacts


## What to write in your README (7 questions)

1. What did I build? 2. How do params / KV-cache / flop-per-token scale?
3. What totals, cache GB, measured slope? 4. What surprised you?
5. Bottleneck at long context? (hint: cache bytes + attention flops, not weights).
6. What if GQA halves kv-heads? int8 cache? batch=8? (recompute one row).
7. Next? (quantization in Session 6 attacks exactly these bytes).
